# 🛍️ Retail Sales Analysis using Pandas

## Objective

Perform exploratory data analysis (EDA) on retail transaction data to investigate:

- Revenue trends
- Customer behaviour
- Product performance
- Order patterns
- Return patterns

In [776]:
import pandas as pd
import numpy as np

## Load Dataset

In [777]:
df = pd.read_csv(
    r"C:\Users\Hp\Downloads\retail_data.csv",
    encoding="cp1252",
    sep=";"
)

print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

   UnitPrice  CustomerID         Country          InvoiceDate TransactionType  
0       2.55       17850  UNITED KINGDOM  2010-12-01 08:26:00        Purchase  
1       3.39       17850  UNITED KINGDOM  2010-12-01 08:26:00        Purchase  
2       2.75       17850  UNITED KINGDOM  2010-12-01 08:26:00        Purchase  
3       3.39       17850  UNITED KINGDOM  2010-12-01 08:26:00        Purchase  
4       3.39       17850  UNITED KINGDOM  2010-12-01 08:26:00        Purchase  


**Observation:**

Dataset loaded successfully.

The dataset contains transaction-level retail sales data including product information, customer identifiers, quantities, pricing, and transaction dates.

## Dataset Overview

In [778]:
df.shape

(51348, 9)

In [779]:
df.dtypes

InvoiceNo              str
StockCode              str
Description            str
Quantity             int64
UnitPrice          float64
CustomerID           int64
Country                str
InvoiceDate            str
TransactionType        str
dtype: object

In [780]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice',
       'CustomerID', 'Country', 'InvoiceDate', 'TransactionType'],
      dtype='str')

**Observation:**

The dataset contains:

- 51,348 records
- 9 columns

Data includes customer, product, pricing, and transaction information suitable for sales analysis.

## Missing Values Analysis

In [781]:
df['CustomerID'] = (
    df['CustomerID']
    .replace(0, np.nan)
)

df.isna().sum()

InvoiceNo              0
StockCode              0
Description          145
Quantity               0
UnitPrice              0
CustomerID         18546
Country                0
InvoiceDate            0
TransactionType        0
dtype: int64

In [782]:
(
    df.isna().sum()
    /
    len(df)
    *
    100
).round(2)

InvoiceNo           0.00
StockCode           0.00
Description         0.28
Quantity            0.00
UnitPrice           0.00
CustomerID         36.12
Country             0.00
InvoiceDate         0.00
TransactionType     0.00
dtype: float64

### Insight

Approximately **36%** of Customer IDs are missing.

Missing customer identifiers may represent:

- Guest purchases
- Incomplete customer records

These values were excluded from customer-level analysis but retained for revenue analysis.

Missing descriptions are minimal and unlikely to impact results significantly.

## Datatype Conversion

In [783]:
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate']
)

df.dtypes

InvoiceNo                     str
StockCode                     str
Description                   str
Quantity                    int64
UnitPrice                 float64
CustomerID                float64
Country                       str
InvoiceDate        datetime64[us]
TransactionType               str
dtype: object

### Insight

InvoiceDate was converted from string to datetime format, enabling time-based analysis including monthly revenue trends.

## Feature Engineering

In [784]:
df['Revenue'] = (
    df['Quantity']
    *
    df['UnitPrice']
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,InvoiceDate,TransactionType,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,15.30
1,536365,71053,WHITE METAL LANTERN,6,3.39,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,20.34


### Insight

Revenue was calculated using:

Revenue = Quantity × UnitPrice

This metric serves as the primary measure across all sales analyses.

## Filter Valid Purchase Transactions

In [785]:
purchase_df = df[
    (
        df['TransactionType']
        ==
        'Purchase'
    )
    &
    (
        df['Quantity']
        >
        0
    )
    &
    (
        df['UnitPrice']
        >
        0
    )
].copy()

purchase_df.head()

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,InvoiceDate,TransactionType,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,15.30
1,536365,71053,WHITE METAL LANTERN,6,3.39,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850.0,UNITED KINGDOM,2010-12-01 08:26:00,Purchase,20.34


**Observation:**

Returns and invalid transactions were excluded.

Only valid purchase records were retained for revenue and customer analyses.

# 📈 Revenue Analysis

In [786]:
total_revenue = (
    purchase_df['Revenue']
    .sum()
)

total_revenue

np.float64(973233.25)

### Insight

Total valid revenue:

**£973k+**

Revenue is strongly concentrated during the holiday season.

## Monthly Revenue Trends

In [787]:
monthly_revenue = (

purchase_df.groupby(
purchase_df[
'InvoiceDate'
]
.dt.month
)[
'Revenue'
]
.sum()

)

print(
monthly_revenue
)

InvoiceDate
1     149487.11
12    823746.14
Name: Revenue, dtype: float64


### Insight

December generated approximately **84%** of total revenue.

This indicates strong seasonal purchasing behaviour and holiday-driven demand.

# 📦 Product Analysis

In [788]:
top_products_revenue = (

purchase_df.groupby(
'Description'
)[
'Revenue'
]
.sum()
.sort_values(
ascending=False
)
.head(
10
)

)

print(
top_products_revenue
)

Description
REGENCY CAKESTAND 3 TIER              31444.62
DOTCOM POSTAGE                        27176.96
WHITE HANGING HEART T-LIGHT HOLDER    13833.23
AMAZON FEE                            13541.33
CHILLI LIGHTS                         11393.56
RED WOOLLY HOTTIE WHITE HEART.         9398.50
PAPER CHAIN KIT 50'S CHRISTMAS         9322.08
WHITE SKULL HOT WATER BOTTLE           8514.51
HOT WATER BOTTLE TEA AND SYMPATHY      8136.18
CHOCOLATE HOT WATER BOTTLE             7627.69
Name: Revenue, dtype: float64


### Insight

**REGENCY CAKESTAND 3 TIER**

was the highest revenue-generating product (~£31k).

Revenue remains distributed across multiple products, reducing dependency risk.

# 👥 Customer Analysis

Unique Customers

In [789]:

purchase_df['CustomerID'].nunique()

990

**Insight:** 
990 unique customers were identified in the datset.

Top Customers

In [790]:
top_customers = (

purchase_df.groupby(
'CustomerID'
)[
'Revenue'
]
.sum()
.sort_values(
ascending=False
)
.head(
10
)

)

print(
top_customers
)


CustomerID
18102.0    27834.61
15061.0    19950.66
16029.0    13202.52
17511.0    10573.22
14646.0     8591.88
14911.0     7995.94
13089.0     7738.67
12415.0     7092.98
16210.0     7000.64
13777.0     6961.78
Name: Revenue, dtype: float64


**Insight:**

The highest-spending customer generated approximately **£27k+** revenue.

A relatively small group of customers contributes a substantial share of total revenue, reflecting wholesale purchasing behaviour.

Average Customer Revenue

In [791]:
customer_totals = (

purchase_df.groupby(
    'CustomerID'
)[
    'Revenue'
]
.sum()

)

customer_totals.mean()

np.float64(704.4968484848484)

**Insight:**

Average revenue per customer was approximately **£704**, indicating high-value wholesale transactions.

Customer Order Frequency

In [792]:
customer_orders = (

purchase_df.groupby(
    'CustomerID'
)[
    'InvoiceNo'
]
.nunique()

)

customer_orders.describe()

count    990.000000
mean       1.663636
std        1.974814
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       37.000000
Name: InvoiceNo, dtype: float64

**Insight:**

Most customers placed a relatively small number of orders, while a smaller segment showed repeated purchasing behaviour.

# 📦 Order Analysis

Unique Orders

In [793]:
unique_orders = (

purchase_df[
    'InvoiceNo'
]
.nunique()

)

unique_orders

1827

**Insight:**

The dataset contains **1,827 unique orders** across **990 customers**, indicating repeat purchasing behaviour among customers.

Average Order Value

In [794]:
invoice_totals = (

purchase_df.groupby(
    'InvoiceNo'
)[
    'Revenue'
]
.sum()

)

avg_order_value = (
    invoice_totals.mean()
)

avg_order_value

np.float64(532.6947181171319)

**Insight:**

Average order value was approximately **£532**, reflecting relatively high-value wholesale purchases rather than individual consumer transactions.

# 🔁 Returns Analysis

In [795]:
returns_df = df[
    df[
        'TransactionType'
    ]
    ==
    'Return'
]

total_returns = len(
    returns_df
)

return_rate = (

    total_returns
    /
    len(df)

) * 100

round(
    return_rate,
    2
)

1.99

**Insight:**

Approximately **2%** of transactions were returns, suggesting relatively low return frequency.

Top Returned Products

In [796]:
top_returns = (

returns_df.groupby(
    'Description'
)[
    'Quantity'
]
.count()
.sort_values(
    ascending=False
)
.head(
    10
)

)

top_returns

Description
REGENCY CAKESTAND 3 TIER            18
RED RETROSPOT CAKE STAND            12
RED RETROSPOT TRADITIONAL TEAPOT    10
MANUAL                              10
DISCOUNT                             9
LARGE POPCORN HOLDER                 9
VICTORIAN SEWING BOX LARGE           9
SILVER HANGING T-LIGHT HOLDER        8
POSTAGE                              8
AMAZON FEE                           8
Name: Quantity, dtype: int64

**Insight:**

Frequently returned products may indicate quality issues, unmet expectations, or ordering mistakes and warrant further investigation.

# ⚠️ Project Limitations

1. **Limited time period**

The dataset covers only **December 2010 to January 2011**, making findings highly influenced by holiday-season purchasing behaviour.

2. **Missing Customer IDs**

Approximately **36%** of Customer IDs are missing, reducing customer-level analysis accuracy.

3. **Revenue seasonality**

Revenue is heavily concentrated in December, potentially limiting generalization across other periods.

4. **Returns excluded**

Return transactions were excluded from revenue calculations to focus on valid sales activity.

# 📌 Final Findings

- Total Revenue: £973k+
- Top Market: United Kingdom
- Peak Month: December 2010
- Top Product: REGENCY CAKESTAND 3 TIER
- Unique Customers: 990
- Average Order Value: £532
- Return Rate: ~2%

## Business Insights

- Revenue strongly concentrated during holiday season
- UK market dominates sales
- High-value customers drive substantial revenue
- Low return rate suggests healthy product-market fit